# RakForest — Download Raw Audio Datasets

โน้ตบุ๊กนี้ดาวน์โหลดเฉพาะชุดข้อมูลที่ใช้ใน model pipeline ปัจจุบัน: ESC-50, Footstep-set, DataSEC v2, RFCx FrugalAI, FSC22, Google Drive Fire Sounds, Vehicle Sounds และชุดเสียงปืนเพิ่มเติม และชุดเสียงคน (VocalSound, Nonspeech7k, Common Voice Thai)

FSD50K ถูกพักไว้ และ Zenodo Fire ถูก quarantine ออกจากการฝึก จึงไม่ดาวน์โหลดโดยค่าเริ่มต้น (เปิดตัวเลือกเฉพาะเมื่อต้องใช้ตรวจข้อมูลด้วยคน) โค้ดจะข้ามชุดที่มีไฟล์เสียงครบอยู่แล้ว

Credentials ของ Kaggle/Hugging Face อ่านจาก `.env` ที่ root ของโปรเจกต์


## 1. ติดตั้ง dependencies

In [43]:
import importlib.util
import subprocess
import sys

# Keep the current environment stable; install only packages that are missing.
required_packages = {
    'kaggle': 'kaggle',
    'datasets': 'datasets[audio]',
    'dotenv': 'python-dotenv',
    'googleapiclient': 'google-api-python-client',
    'google.auth': 'google-auth',
    'requests': 'requests',
}
missing_packages = [package for module, package in required_packages.items()
                    if importlib.util.find_spec(module) is None]
if missing_packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
else:
    print('Download dependencies are already installed; no package updates needed.')


Download dependencies are already installed; no package updates needed.


## 2. ตั้งค่าโปรเจกต์และโหลด credentials
ไฟล์ `.env` ใช้เก็บ `KAGGLE_USERNAME`, `KAGGLE_KEY`, `HF_TOKEN` และ `MDC_API_KEY` (เฉพาะ Common Voice; ต้องยอมรับเงื่อนไขชุดข้อมูลบน Mozilla Data Collective และสร้าง API key ก่อน)


In [44]:
from pathlib import Path
import os
from dotenv import load_dotenv
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks': PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)
load_dotenv(PROJECT_ROOT / '.env')
missing = [k for k in ('KAGGLE_USERNAME', 'KAGGLE_KEY') if not os.getenv(k)]
if missing: raise RuntimeError(f'กรุณาใส่ {missing} ใน .env ก่อนดาวน์โหลด')
print('Project root:', PROJECT_ROOT)
print('Kaggle credentials loaded:', True)
print('Hugging Face token available:', bool(os.getenv('HF_TOKEN')))

Project root: /Users/guidegiegg/งาน/โปรเจคเล็กๆ/RakForest---AI-Powered-Abnormal-Forest-Sound-Detection
Kaggle credentials loaded: True
Hugging Face token available: True


## 3. ตัวช่วยดาวน์โหลดและแตก ZIP
ใช้ข้ามไฟล์ที่มีอยู่แล้ว และดาวน์โหลดแบบ streaming เพื่อไม่โหลดไฟล์ทั้งก้อนเข้า RAM

In [45]:
import requests
import zipfile

AUDIO_EXTENSIONS = {'.wav', '.mp3', '.flac', '.ogg', '.m4a', '.aac', '.wma', '.aiff', '.aif'}

def has_audio(folder):
    folder = Path(folder)
    return folder.exists() and any(
        p.is_file() and p.suffix.lower() in AUDIO_EXTENSIONS
        for p in folder.rglob('*')
    )

def download_file(url, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        print(f'[skip download] {destination.name}')
        return destination
    partial = destination.with_suffix(destination.suffix + '.part')
    existing = partial.stat().st_size if partial.exists() else 0
    headers = {'Range': f'bytes={existing}-'} if existing else {}
    with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        append = existing > 0 and response.status_code == 206
        if existing and not append:
            existing = 0
        total = response.headers.get('Content-Range', '').rsplit('/', 1)[-1]
        if not total or total == '*':
            content_length = response.headers.get('Content-Length')
            total = str(existing + int(content_length)) if content_length else ''
        total = int(total) if total.isdigit() else 0
        mode = 'ab' if append else 'wb'
        downloaded = existing
        last_reported = downloaded // (100 * 1024 * 1024)
        print(f'[download] {destination.name}: resumed at {downloaded / (1024**2):.1f} MiB' if append else f'[download] {destination.name}')
        with partial.open(mode) as f:
            for chunk in response.iter_content(8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    current_report = downloaded // (100 * 1024 * 1024)
                    if current_report > last_reported:
                        last_reported = current_report
                        if total:
                            print(f'  {destination.name}: {downloaded / total:.0%} ({downloaded / (1024**3):.2f}/{total / (1024**3):.2f} GiB)')
                        else:
                            print(f'  {destination.name}: {downloaded / (1024**3):.2f} GiB')
    partial.replace(destination)
    print(f'[downloaded] {destination.name}: {destination.stat().st_size / (1024**3):.2f} GiB')
    return destination

def extract_zip(archive, destination):
    destination = Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    if has_audio(destination):
        print(f'[skip extract] audio already exists in {destination}')
        return
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(destination)
    print(f'[extracted] {destination}')


## 4. ESC-50 (GitHub)
ดาวน์โหลดไฟล์ ZIP จาก repository ทางการแล้วแตกไว้ใน `data/raw/esc50/`.

In [46]:
ESC50_DIR = RAW_DIR / 'esc50'
if has_audio(ESC50_DIR):
    print('[skip] ESC-50: พบไฟล์เสียงแล้ว')
else:
    esc_zip = download_file('https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip', RAW_DIR / 'archives' / 'esc50.zip')
    extract_zip(esc_zip, ESC50_DIR)


[skip] ESC-50: พบไฟล์เสียงแล้ว


## 5. Footstep-set (Zenodo)
ดาวน์โหลดชุด Footstep-set เท่านั้น (ไม่โหลด Impact-set ที่อยู่ record เดียวกัน).

In [47]:
FOOTSTEP_DIR = RAW_DIR / 'footstep_set'
if has_audio(FOOTSTEP_DIR):
    print('[skip] Footstep-set: พบไฟล์เสียงแล้ว')
else:
    foot_zip = download_file('https://zenodo.org/records/14286414/files/Footstep-set.zip?download=1', RAW_DIR / 'archives' / 'Footstep-set.zip')
    extract_zip(foot_zip, FOOTSTEP_DIR)


[skip] Footstep-set: พบไฟล์เสียงแล้ว


## 6. DataSEC v2 (Zenodo)
ไฟล์บีบอัดประมาณ 6.4 GB; ตรวจสอบพื้นที่ว่างก่อนเริ่ม.

In [48]:
DATASEC_DIR = RAW_DIR / 'datasec_v2'
LEGACY_DATASEC_DIR = RAW_DIR / 'datasec_vehicle_v2'
if not has_audio(DATASEC_DIR) and has_audio(LEGACY_DATASEC_DIR):
    DATASEC_DIR = LEGACY_DATASEC_DIR
if has_audio(DATASEC_DIR):
    print('[skip] DataSEC v2: พบไฟล์เสียงแล้ว')
else:
    datasec_zip = download_file('https://zenodo.org/records/15393250/files/DATASEC.zip?download=1', RAW_DIR / 'archives' / 'DATASEC.zip')
    extract_zip(datasec_zip, DATASEC_DIR)


[skip] DataSEC v2: พบไฟล์เสียงแล้ว


## 7. RFCx / FrugalAI (Hugging Face)
ต้องมี `HF_TOKEN` ใน `.env` และยอมรับเงื่อนไขการเข้าถึง dataset บน Hugging Face ก่อน

ถ้าเห็น `TqdmWarning: IProgress not found` ระหว่างดาวน์โหลด เป็นเพียง warning ของ progress bar; การดาวน์โหลดมักทำต่อได้ตามปกติ. มี `ipywidgets` ใน cell ติดตั้ง dependencies สำหรับการรันครั้งถัดไปแล้ว ไม่ต้องรันติดตั้งซ้ำหรือ restart kernel ระหว่าง RFCx กำลังโหลด


ดาวน์โหลดและบันทึก Hugging Face dataset ไว้ใน `data/raw/rfcx_frugalai/`.


In [49]:
from datasets import load_dataset
RFCX_DIR = RAW_DIR / 'rfcx_frugalai' / 'hf_dataset'
if RFCX_DIR.exists() and any(RFCX_DIR.iterdir()):
    print('[skip] RFCx FrugalAI: พบข้อมูลอยู่แล้ว')
else:
    RFCX_DIR.parent.mkdir(parents=True, exist_ok=True)
    rfcx = load_dataset('rfcx/frugalai', token=os.getenv('HF_TOKEN') or None, cache_dir=str(RFCX_DIR.parent / '.hf_cache'))
    rfcx.save_to_disk(str(RFCX_DIR))
    print('[done] RFCx FrugalAI:', RFCX_DIR)

[skip] RFCx FrugalAI: พบข้อมูลอยู่แล้ว


## 8. FSC22 และ Vehicle Sounds (Kaggle)
ใช้ Kaggle API credentials จาก `.env`; ดาวน์โหลดสองชุดตามรายการที่ให้มา.

In [50]:
KAGGLE_DATASETS = {
    'fsc22': ('irmiot22/fsc22-dataset', RAW_DIR / 'fsc22'),
    'vehicle_sounds': ('janboubiabderrahim/vehicle-sounds-dataset', RAW_DIR / 'vehicle_sounds'),
}
for name, (dataset_id, destination) in KAGGLE_DATASETS.items():
    destination.mkdir(parents=True, exist_ok=True)
    if has_audio(destination):
        print(f'[skip] {name}: มีไฟล์เสียงอยู่แล้ว')
        continue
    print(f'[download] {name}: {dataset_id}')
    subprocess.run([sys.executable, '-m', 'kaggle', 'datasets', 'download', '-d', dataset_id, '-p', str(destination), '--unzip'], check=True, cwd=PROJECT_ROOT)


[skip] fsc22: มีไฟล์เสียงอยู่แล้ว
[skip] vehicle_sounds: มีไฟล์เสียงอยู่แล้ว


## 9. FSD50K v1.0 (Zenodo)
ชุดนี้ใหญ่มากและประกอบด้วยไฟล์ ZIP แบบแบ่ง part; cell นี้ดึงรายการไฟล์จาก Zenodo API และดาวน์โหลดทุกไฟล์ของ record ก่อนประกอบ archive. ต้องมีพื้นที่ว่างหลายสิบ GB.

In [51]:
USE_FSD50K = False  # เปิดเป็น True หากต้องการดาวน์โหลด/แตกไฟล์ FSD50K อีกครั้ง
if not USE_FSD50K:
    print('[paused] FSD50K: ข้ามการดาวน์โหลดและแตกไฟล์')
else:
    import requests
    import subprocess
    FSD_DIR = RAW_DIR / 'fsd50k_v1'
    FSD_ARCHIVES = RAW_DIR / 'archives' / 'fsd50k_v1'
    FSD_DIR.mkdir(parents=True, exist_ok=True)
    FSD_ARCHIVES.mkdir(parents=True, exist_ok=True)
    if has_audio(FSD_DIR):
        print('[skip] FSD50K: พบไฟล์เสียงแล้ว')
    else:
        record = requests.get('https://zenodo.org/api/records/4060432', timeout=60).json()
        files = record.get('files', [])
        selected = [f for f in files if f.get('key', '').lower().endswith(('.zip', '.z01', '.z02', '.z03', '.z04', '.z05'))]
        print('Archive files:', len(selected))
        for item in selected:
            download_file(item['links']['self'], FSD_ARCHIVES / item['key'])
        # Merge split archives using the system zip utility, then extract each complete ZIP.
        for prefix in ('FSD50K.dev_audio', 'FSD50K.eval_audio'):
            parts = sorted(FSD_ARCHIVES.glob(prefix + '.z*'))
            main_zip = FSD_ARCHIVES / (prefix + '.zip')
            if main_zip.exists() and parts:
                merged = FSD_ARCHIVES / (prefix + '_merged.zip')
                subprocess.run(['zip', '-s', '0', str(main_zip), '--out', str(merged)], check=True)
                with zipfile.ZipFile(merged) as zf: zf.extractall(FSD_DIR)
            elif main_zip.exists():
                with zipfile.ZipFile(main_zip) as zf: zf.extractall(FSD_DIR)
        for key in ('FSD50K.ground_truth.zip', 'FSD50K.metadata.zip', 'FSD50K.doc.zip'):
            archive = FSD_ARCHIVES / key
            if archive.exists():
                with zipfile.ZipFile(archive) as zf: zf.extractall(FSD_DIR)
        print('[done] FSD50K:', FSD_DIR)


[paused] FSD50K: ข้ามการดาวน์โหลดและแตกไฟล์


## 10. Google Drive Fire Sounds
ลองดาวน์โหลดจากลิงก์แชร์โดยตรงก่อน หากโฟลเดอร์จำกัดสิทธิ์ โค้ดจะลองใช้ Google ADC ที่ตั้งไว้ในเครื่องเป็นทางสำรอง

หากโฟลเดอร์ Drive ตั้งเป็น “ทุกคนที่มีลิงก์ดูได้” ให้รัน cell ถัดไปได้เลย ไม่ต้องตั้งค่า ADC หากเป็นโฟลเดอร์ส่วนตัว cell จะลองใช้ ADC และแจ้งวิธีตั้งค่าเมื่อจำเป็น

In [52]:
import importlib.util
import json
import subprocess
import sys

if importlib.util.find_spec('gdown') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'gdown'])
import gdown

DRIVE_FOLDER_ID = '1QOYouznSh3YZR_6tbDPDQIHNyLorzqXN'
DRIVE_DIR = RAW_DIR / 'fire_google_drive'
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
DOWNLOAD_MARKER = DRIVE_DIR / '.download_complete.json'

def find_drive_wavs():
    return sorted(p for p in DRIVE_DIR.rglob('*') if p.is_file() and p.suffix.lower() == '.wav')

wav_files = find_drive_wavs()
marker_data = {}
if DOWNLOAD_MARKER.exists():
    try:
        marker_data = json.loads(DOWNLOAD_MARKER.read_text())
    except Exception:
        marker_data = {}

if marker_data.get('folder_id') == DRIVE_FOLDER_ID and marker_data.get('wav_count') == len(wav_files) and wav_files:
    print(f'[skip] ดาวน์โหลดครบแล้วตาม marker: {len(wav_files):,} WAV')
else:
    print(f'พบ WAV เดิม {len(wav_files):,} ไฟล์; จะตรวจ/ดาวน์โหลดโฟลเดอร์ที่แชร์ให้ครบก่อนทำเครื่องหมายเสร็จ')
    public_download_ok = False
    try:
        gdown.download_folder(
            id=DRIVE_FOLDER_ID,
            output=str(DRIVE_DIR),
            quiet=False,
            use_cookies=False,
        )
        public_download_ok = True
    except Exception as e:
        print(f'ดาวน์โหลดแบบลิงก์แชร์ไม่สำเร็จ: {type(e).__name__}: {e}')

    wav_files = find_drive_wavs()
    if not public_download_ok:
        # If public link download failed, try Google Application Default Credentials.
        try:
            from google.auth import default
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            credentials, _ = default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
        except Exception as e:
            raise RuntimeError(
                'ยังดาวน์โหลด WAV ไม่ได้: โฟลเดอร์ควรแชร์เป็น Anyone with the link (Viewer) '
                'ถ้ายังเข้าไม่ได้ ให้รัน `gcloud auth application-default login '
                '--scopes=https://www.googleapis.com/auth/drive.readonly` ใน Terminal '
                'ด้วยบัญชีที่มีสิทธิ์อ่านโฟลเดอร์ แล้วรัน cell นี้ใหม่'
            ) from e

        drive = build('drive', 'v3', credentials=credentials, cache_discovery=False)
        items, page_token = [], None
        while True:
            result = drive.files().list(
                q=f"'{DRIVE_FOLDER_ID}' in parents and trashed = false",
                pageSize=1000, pageToken=page_token,
                fields='nextPageToken, files(id,name,mimeType,size)',
                includeItemsFromAllDrives=True, supportsAllDrives=True,
            ).execute()
            items.extend(result.get('files', []))
            page_token = result.get('nextPageToken')
            if not page_token:
                break
        wav_items = [f for f in items if f.get('mimeType') == 'audio/wav' or f.get('name', '').lower().endswith('.wav')]
        if not wav_items:
            raise RuntimeError('เชื่อมต่อ Drive ได้แล้ว แต่ไม่พบ WAV ในโฟลเดอร์ที่ระบุ หรือบัญชีนี้ไม่มีสิทธิ์อ่าน')
        for i, item in enumerate(wav_items, 1):
            out = DRIVE_DIR / Path(item['name']).name
            expected = int(item.get('size') or 0)
            if out.exists() and (not expected or out.stat().st_size == expected):
                continue
            request = drive.files().get_media(fileId=item['id'], supportsAllDrives=True)
            with out.open('wb') as fh:
                downloader = MediaIoBaseDownload(fh, request, chunksize=8 * 1024 * 1024)
                done = False
                while not done:
                    _, done = downloader.next_chunk()
            if i % 25 == 0 or i == len(wav_items):
                print(f'Downloaded {i}/{len(wav_items)}')
        wav_files = find_drive_wavs()

    if not wav_files:
        raise RuntimeError('ไม่พบไฟล์ WAV หลังดาวน์โหลด; ตรวจสิทธิ์และโครงสร้างโฟลเดอร์ Drive')
    DOWNLOAD_MARKER.write_text(json.dumps({'folder_id': DRIVE_FOLDER_ID, 'wav_count': len(wav_files)}, indent=2))

print(f'[done] Google Drive WAV: {len(wav_files):,} files in {DRIVE_DIR}')

[skip] ดาวน์โหลดครบแล้วตาม marker: 721 WAV
[done] Google Drive WAV: 721 files in /Users/guidegiegg/งาน/โปรเจคเล็กๆ/RakForest---AI-Powered-Abnormal-Forest-Sound-Detection/data/raw/fire_google_drive


## 11. Zenodo Fire (optional, review-only)
ชุดนี้ถูก quarantine จาก model training และมีขนาดใหญ่ จึงปิดดาวน์โหลดไว้ก่อน เปิด flag ใน cell ถัดไปเฉพาะเมื่อต้องใช้ตรวจข้อมูลด้วยคน


In [53]:
DOWNLOAD_ZENODO_FIRE_FOR_REVIEW = False  # review-only; not used to train the current models
ZENODO_FIRE_DIR = RAW_DIR / 'zenodo_fire'
if not DOWNLOAD_ZENODO_FIRE_FOR_REVIEW:
    print('[paused] Zenodo Fire: ไม่ดาวน์โหลด เพราะถูก quarantine จาก model training')
elif has_audio(ZENODO_FIRE_DIR):
    print('[skip] Zenodo Fire: พบไฟล์เสียงแล้ว')
else:
    fire_zip = download_file('https://zenodo.org/records/20132520/files/data.zip?download=1', RAW_DIR / 'archives' / 'zenodo_fire_data.zip')
    extract_zip(fire_zip, ZENODO_FIRE_DIR)


[paused] Zenodo Fire: ไม่ดาวน์โหลด เพราะถูก quarantine จาก model training


## 12. ชุดเสียงปืนเพิ่มเติม (ดาวน์โหลดสาธารณะ)

ดาวน์โหลดแยกจากชุดเดิมไว้ใน `data/raw/` โดยเลือกเฉพาะเสียงปืน: C3GD, Tropical Forest Gunshot, Zenodo Gunshot และไฟล์ `gunshot.zip` ของ M16 (ไม่ดาวน์โหลด firecracker) จากนั้นสร้าง manifest ตรวจไฟล์ซ้ำด้วย SHA-256 และรหัสเหตุการณ์/ไมค์ของ C3GD โดยไม่ลบไฟล์ต้นฉบับ


In [54]:
import concurrent.futures
import hashlib
import time
import shutil
import pandas as pd

def checksum_file(path, algorithm='sha256'):
    h = hashlib.new(algorithm)
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

GUNSHOT_EXTRA_DIRS = {
    'c3gd': RAW_DIR / 'c3gd',
    'tropical_forest_gunshot': RAW_DIR / 'tropical_forest_gunshot',
    'zenodo_gunshot': RAW_DIR / 'zenodo_gunshot',
    'm16_gunshots': RAW_DIR / 'm16_gunshots',
}
ARCHIVE_DIR = RAW_DIR / 'archives'
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

# Use Zenodo's public record API to discover files/checksums, then download only the dataset archives.
def download_zenodo_record_file(record_id, filename, archive_path):
    api = requests.get(f'https://zenodo.org/api/records/{record_id}', timeout=60)
    api.raise_for_status()
    record = api.json()
    item = next((f for f in record.get('files', []) if f.get('key') == filename), None)
    if item is None:
        raise RuntimeError(f'ไม่พบ {filename} ใน Zenodo record {record_id}')
    path = Path(archive_path)
    expected_size = int(item.get('size') or 0)
    if path.exists() and expected_size and path.stat().st_size != expected_size:
        path.unlink()
    if not path.exists():
        download_file(item['links']['self'], path)
    if expected_size and path.stat().st_size != expected_size:
        raise RuntimeError(f'ขนาดไฟล์ไม่ตรง metadata: {path.name}')
    checksum = item.get('checksum', '')
    if checksum.startswith('md5:') and checksum_file(path, 'md5') != checksum.split(':', 1)[1]:
        path.unlink()
        raise RuntimeError(f'MD5 ไม่ตรง metadata: {path.name}')
    return path

# C3GD: 12,112 outdoor live-fire WAV clips; archive is about 771 MB.
C3GD_DIR = GUNSHOT_EXTRA_DIRS['c3gd']
if has_audio(C3GD_DIR):
    print('[skip] C3GD: พบไฟล์เสียงแล้ว')
else:
    c3gd_zip = download_zenodo_record_file('22286299', 'C3GD.zip', ARCHIVE_DIR / 'C3GD.zip')
    extract_zip(c3gd_zip, C3GD_DIR)

# Tropical forest data: fetch only the 597 train + 150 validation gunshot WAVs.
# Public Mendeley metadata identifies these two Gunshot folders by their published counts.
TROP_DIR = GUNSHOT_EXTRA_DIRS['tropical_forest_gunshot']
TROP_COMPLETE = TROP_DIR / '.download_complete.json'
if TROP_COMPLETE.exists() and has_audio(TROP_DIR):
    print('[skip] Tropical Forest Gunshot: พบชุดที่ดาวน์โหลดครบแล้ว')
else:
    meta = requests.get('https://data.mendeley.com/public-api/datasets/x48cwz364j', timeout=60)
    meta.raise_for_status()
    trop = meta.json()
    wav_entries = [f for f in trop.get('files', []) if f.get('filename', '').lower().endswith('.wav')]
    by_folder = {}
    for item in wav_entries:
        by_folder.setdefault(item.get('folder_id'), []).append(item)
    gun_folders = [items for items in by_folder.values() if len(items) in (597, 150)]
    if sorted(len(items) for items in gun_folders) != [150, 597]:
        raise RuntimeError('โครงสร้าง/จำนวนไฟล์ Gunshot ของ Mendeley เปลี่ยนไป; หยุดไว้เพื่อไม่ดาวน์โหลด Background โดยผิดพลาด')
    selected = []
    for items in gun_folders:
        split_name = 'train' if len(items) == 597 else 'validation'
        selected.extend((split_name, item) for item in items)
    TROP_DIR.mkdir(parents=True, exist_ok=True)
    def fetch_tropical(pair):
        split_name, item = pair
        out = TROP_DIR / split_name / item['filename']
        out.parent.mkdir(parents=True, exist_ok=True)
        details = item.get('content_details', {})
        expected_hash = details.get('sha256_hash')
        expected_size = int(item.get('size') or details.get('size') or 0)
        if out.exists():
            size_ok = not expected_size or out.stat().st_size == expected_size
            hash_ok = not expected_hash or checksum_file(out) == expected_hash
            if size_ok and hash_ok: return out
            out.unlink()
        url = details.get('download_url')
        if not url: raise RuntimeError(f"ไม่มี download_url สำหรับ {item['filename']}")
        partial = out.with_suffix(out.suffix + '.part')
        # Promote a complete partial file left by an interrupted notebook run.
        if partial.exists() and expected_size and partial.stat().st_size == expected_size:
            if not expected_hash or checksum_file(partial) == expected_hash:
                partial.replace(out)
                return out
        if partial.exists() and expected_size and partial.stat().st_size > expected_size:
            partial.unlink()
        last_error = None
        for attempt in range(1, 6):
            try:
                offset = partial.stat().st_size if partial.exists() else 0
                headers = {'Range': f'bytes={offset}-'} if offset else {}
                with requests.get(url, headers=headers, stream=True, timeout=(30, 240)) as response:
                    # Some object stores ignore Range. In that case restart this file safely.
                    if offset and response.status_code == 200:
                        mode = 'wb'
                    else:
                        response.raise_for_status()
                        if offset and response.status_code != 206:
                            raise RuntimeError(f"เซิร์ฟเวอร์ไม่รองรับการต่อไฟล์ ({response.status_code})")
                        mode = 'ab' if offset else 'wb'
                    with partial.open(mode) as f:
                        for chunk in response.iter_content(1024 * 1024):
                            if chunk:
                                f.write(chunk)
                if expected_size and partial.stat().st_size != expected_size:
                    raise ConnectionError(f"ขนาดยังไม่ครบ ({partial.stat().st_size}/{expected_size} bytes)")
                if expected_hash and checksum_file(partial) != expected_hash:
                    partial.unlink(missing_ok=True)
                    raise RuntimeError(f"SHA-256 ไม่ตรง: {item['filename']}")
                partial.replace(out)
                return out
            except Exception as exc:
                last_error = exc
                if attempt < 5:
                    wait_seconds = min(2 ** (attempt - 1), 16)
                    print(f"[retry {attempt}/5] {item['filename']}: {exc}; รอ {wait_seconds}s แล้วลองต่อ")
                    time.sleep(wait_seconds)
        raise RuntimeError(f"ดาวน์โหลด {item['filename']} ไม่สำเร็จหลังลอง 5 ครั้ง; เก็บไฟล์ .part ไว้เพื่อรันต่อได้: {last_error}")
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
        for i, _ in enumerate(pool.map(fetch_tropical, selected), 1):
            if i % 100 == 0 or i == len(selected): print(f'[Tropical Gunshot] {i}/{len(selected)}')
    TROP_COMPLETE.write_text(json.dumps({'dataset_id':'x48cwz364j','version':trop.get('version'),'gunshot_wav_count':len(selected)}, indent=2))

# Zenodo Gunshot/Gunfire: download the public 1.57 GB archive and extract its WAVs.
ZEN_GUN_DIR = GUNSHOT_EXTRA_DIRS['zenodo_gunshot']
if has_audio(ZEN_GUN_DIR):
    print('[skip] Zenodo Gunshot: พบไฟล์เสียงแล้ว')
else:
    zen_zip = download_zenodo_record_file('7004819', 'edge-collected-gunshot-audio.zip', ARCHIVE_DIR / 'edge-collected-gunshot-audio.zip')
    extract_zip(zen_zip, ZEN_GUN_DIR)

# M16 dataset: download only gunshot.zip; firecracker.zip is intentionally excluded.
M16_DIR = GUNSHOT_EXTRA_DIRS['m16_gunshots']
if has_audio(M16_DIR):
    print('[skip] M16 Gunshots: พบไฟล์เสียงแล้ว')
else:
    m16_api = requests.get('https://data.mendeley.com/public-api/datasets/kbgw323248', timeout=60)
    m16_api.raise_for_status()
    m16_meta = m16_api.json()
    m16_zip_info = next((f for f in m16_meta.get('files', []) if f.get('filename') == 'gunshot.zip'), None)
    if not m16_zip_info: raise RuntimeError('ไม่พบ gunshot.zip ใน Mendeley M16 dataset')
    m16_zip = ARCHIVE_DIR / 'M16_gunshot.zip'
    expected_size = int(m16_zip_info.get('size') or 0)
    if m16_zip.exists() and expected_size and m16_zip.stat().st_size != expected_size: m16_zip.unlink()
    if not m16_zip.exists(): download_file(m16_zip_info['content_details']['download_url'], m16_zip)
    if expected_size and m16_zip.stat().st_size != expected_size: raise RuntimeError('ขนาด M16 gunshot.zip ไม่ตรง metadata')
    m16_sha = m16_zip_info.get('content_details', {}).get('sha256_hash')
    if m16_sha and checksum_file(m16_zip) != m16_sha:
        m16_zip.unlink()
        raise RuntimeError('SHA-256 ของ M16 gunshot.zip ไม่ตรง metadata')
    extract_zip(m16_zip, M16_DIR)

source_manifest = {
    'c3gd': {'url':'https://github.com/Stonewall-Defense/C3GD', 'license':'CC BY 4.0', 'selected_content':'live-fire WAV clips; deduplicate same event across microphones'},
    'tropical_forest_gunshot': {'url':'https://data.mendeley.com/datasets/x48cwz364j/1', 'license':'CC BY 4.0', 'selected_content':'Gunshot WAV folders only; background excluded'},
    'zenodo_gunshot': {'url':'https://zenodo.org/records/7004819', 'license':'not specified on the record page; verify before redistribution or publication', 'selected_content':'gunshot/gunfire recordings; background may be present'},
    'm16_gunshots': {'url':'https://data.mendeley.com/datasets/kbgw323248/1', 'license':'CC BY 4.0', 'selected_content':'gunshot.zip only; firecracker.zip excluded'},
}
source_manifest_path = PROJECT_ROOT / 'data' / 'manifests' / 'gunshot_extra_sources.json'
source_manifest_path.parent.mkdir(parents=True, exist_ok=True)
source_manifest_path.write_text(json.dumps(source_manifest, indent=2, ensure_ascii=False))
print('Gunshot source download/extraction stage complete.')

[skip] C3GD: พบไฟล์เสียงแล้ว
[skip] Tropical Forest Gunshot: พบชุดที่ดาวน์โหลดครบแล้ว
[skip] Zenodo Gunshot: พบไฟล์เสียงแล้ว
[skip] M16 Gunshots: พบไฟล์เสียงแล้ว
Gunshot source download/extraction stage complete.


## 13. ชุดเสียงคนเพิ่มเติม

ดาวน์โหลดและแยกเก็บใน `data/raw/` โดยรักษา split เดิมไว้: VocalSound (train/valid/test), Nonspeech7k (train/test) และ Common Voice Thai 27.0. Common Voice มีขนาดประมาณ 8.38 GB; ต้องยอมรับเงื่อนไขบน Mozilla Data Collective, สร้าง API key และเพิ่ม `MDC_API_KEY` ลง `.env` ก่อนรัน ส่วนชุด Zenodo ดาวน์โหลดได้ตรงจาก record API.


In [55]:
import tarfile

HUMAN_AUDIO_DIRS = {
    'vocalsound': RAW_DIR / 'vocalsound',
    'nonspeech7k': RAW_DIR / 'nonspeech7k',
    'common_voice_thai_27': RAW_DIR / 'common_voice_thai_27',
}

# Safe extraction: keep only audio and annotation/metadata files, reject links and path traversal.
def extract_tar_selected(archive, destination, allowed_suffixes):
    archive, destination = Path(archive), Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    extracted = 0
    with tarfile.open(archive, 'r:*') as tf:
        for member in tf.getmembers():
            if not member.isfile() or Path(member.name).suffix.lower() not in allowed_suffixes:
                continue
            out = (destination / member.name).resolve()
            if out != root and root not in out.parents:
                raise RuntimeError(f'พบ path ที่ออกนอกโฟลเดอร์ใน archive: {member.name}')
            out.parent.mkdir(parents=True, exist_ok=True)
            src = tf.extractfile(member)
            if src is None: continue
            with src, out.open('wb') as dst: shutil.copyfileobj(src, dst)
            extracted += 1
    return extracted

# VocalSound: 60 published WebDataset tar shards, preserving train/valid/test split.
VOCALSOUND_DIR = HUMAN_AUDIO_DIRS['vocalsound']
VOCALSOUND_DONE = VOCALSOUND_DIR / '.extracted_shards'
VOCALSOUND_DONE.mkdir(parents=True, exist_ok=True)
if not has_audio(VOCALSOUND_DIR):
    vs_record = requests.get('https://zenodo.org/api/records/14650192', timeout=60)
    vs_record.raise_for_status()
    vs_files = [f for f in vs_record.json().get('files', []) if f.get('key','').startswith('wds-audio-') and f.get('key','').endswith('.tar')]
    if len(vs_files) != 60:
        raise RuntimeError(f'VocalSound shard count changed (expected 60, got {len(vs_files)}); inspect the record before downloading')
    for index, item in enumerate(sorted(vs_files, key=lambda x:x['key']), 1):
        shard = ARCHIVE_DIR / 'vocalsound' / item['key']
        marker = VOCALSOUND_DONE / (item['key'] + '.done')
        shard = download_zenodo_record_file('14650192', item['key'], shard)
        if not marker.exists():
            split_name = item['key'].split('-')[2]
            count = extract_tar_selected(shard, VOCALSOUND_DIR / split_name, {'.wav','.json'})
            if count == 0: raise RuntimeError(f'ไม่พบ WAV/JSON ใน {item["key"]}')
            marker.write_text(str(count))
        if index % 5 == 0 or index == len(vs_files): print(f'[VocalSound] {index}/{len(vs_files)} shards')
else:
    print('[skip] VocalSound: พบไฟล์เสียงแล้ว')

# Nonspeech7k: WAVs plus CSV metadata; retain the published train/test folders.
NONSPEECH_DIR = HUMAN_AUDIO_DIRS['nonspeech7k']
if not has_audio(NONSPEECH_DIR / 'train'):
    train_zip = download_zenodo_record_file('6967442', 'train.zip', ARCHIVE_DIR / 'nonspeech7k_train.zip')
    extract_zip(train_zip, NONSPEECH_DIR / 'train')
else: print('[skip] Nonspeech7k train: พบไฟล์เสียงแล้ว')
if not has_audio(NONSPEECH_DIR / 'test'):
    test_zip = download_zenodo_record_file('6967442', 'test.zip', ARCHIVE_DIR / 'nonspeech7k_test.zip')
    extract_zip(test_zip, NONSPEECH_DIR / 'test')
else: print('[skip] Nonspeech7k test: พบไฟล์เสียงแล้ว')
NONSPEECH_META = NONSPEECH_DIR / 'metadata'
NONSPEECH_META.mkdir(parents=True, exist_ok=True)
for filename in ('metadata of train set .csv', 'metadata of test set.csv', 'youtube ID vs link .TXT'):
    if not (NONSPEECH_META / filename).exists():
        meta_file = download_zenodo_record_file('6967442', filename, ARCHIVE_DIR / 'nonspeech7k' / filename)
        shutil.copy2(meta_file, NONSPEECH_META / filename)

# Common Voice Thai 27.0 is served by Mozilla Data Collective. Access terms must be accepted in its UI first.
# Store MDC_API_KEY in the project .env; never paste the key into this notebook.
COMMON_VOICE_DIR = HUMAN_AUDIO_DIRS['common_voice_thai_27']
COMMON_VOICE_ARCHIVES = ARCHIVE_DIR / 'common_voice_thai_27'
COMMON_VOICE_ARCHIVES.mkdir(parents=True, exist_ok=True)
COMMON_VOICE_MARKER = COMMON_VOICE_DIR / '.extract_complete.json'
MDC_API_KEY = os.getenv('MDC_API_KEY')
if has_audio(COMMON_VOICE_DIR) and COMMON_VOICE_MARKER.exists():
    print('[skip] Common Voice Thai 27.0: พบชุดที่แตกไฟล์ครบแล้ว')
elif not MDC_API_KEY:
    print('[needs setup] Common Voice Thai: ยอมรับ terms และสร้าง API key ที่ Mozilla Data Collective แล้วเพิ่ม MDC_API_KEY ใน .env จากนั้นรัน cell นี้ซ้ำ')
else:
    dataset_id = 'cmu5x5tj000f2nq0755h8qcy5'
    api_headers = {'Authorization': f'Bearer {MDC_API_KEY}'}
    try:
        grant = requests.post(f'https://mozilladatacollective.com/api/datasets/{dataset_id}/download', headers=api_headers, timeout=60)
        if grant.status_code == 403:
            raise RuntimeError('บัญชียังไม่ยอมรับ terms/ไม่ผ่านข้อกำหนดการเข้าถึง; เปิดหน้าชุดข้อมูลบน Mozilla Data Collective เพื่อยอมรับก่อน')
        if grant.status_code == 401:
            raise RuntimeError('MDC_API_KEY ไม่ถูกต้อง; สร้าง API key ใหม่ใน Mozilla Data Collective')
        grant.raise_for_status()
        grant_info = grant.json()
        cv_filename = grant_info['filename']
        cv_archive = download_file(grant_info['downloadUrl'], COMMON_VOICE_ARCHIVES / cv_filename)
        expected_bytes = int(grant_info.get('sizeBytes') or 0)
        if expected_bytes and cv_archive.stat().st_size != expected_bytes:
            raise RuntimeError('ขนาด archive Common Voice ไม่ตรงกับ metadata')
        expected_sha = grant_info.get('checksum')
        if expected_sha and checksum_file(cv_archive, 'sha256') != expected_sha:
            raise RuntimeError('SHA-256 ของ Common Voice ไม่ตรงกับ metadata')
        if not has_audio(COMMON_VOICE_DIR):
            extracted_count = extract_tar_selected(cv_archive, COMMON_VOICE_DIR, {'.mp3','.tsv','.txt'})
            if extracted_count == 0: raise RuntimeError('ไม่พบ MP3/metadata ใน Common Voice archive')
            COMMON_VOICE_MARKER.write_text(json.dumps({'dataset_id':dataset_id,'filename':cv_filename,'size_bytes':cv_archive.stat().st_size,'extracted_members':extracted_count}, indent=2))
        print(f'[done] Common Voice Thai 27.0: {COMMON_VOICE_DIR}')
    except Exception as e:
        print(f'[paused] Common Voice Thai: {type(e).__name__}: {e}')
        print('ชุด VocalSound และ Nonspeech7k ดาวน์โหลดแยกได้ตามปกติ; ตั้งค่า terms/API key แล้วค่อยรัน Common Voice cell ซ้ำ')

human_sources = {
    'vocalsound': {'url':'https://zenodo.org/records/14650192','license':'CC BY-SA (per VocalSound project page)','note':'21,024 vocal sound clips; retain train/valid/test; tar shards contain WAV plus JSON labels'},
    'nonspeech7k': {'url':'https://zenodo.org/records/6967442','license':'CC BY-NC-SA 4.0; academic/non-commercial use only','note':'7,014 non-speech human vocalization WAVs; retain train/test split and metadata'},
    'common_voice_thai_27': {'url':'https://mozilladatacollective.com/datasets/cmu5x5tj000f2nq0755h8qcy5','license':'CC0-1.0; do not identify speakers or re-host/re-share dataset','note':'Thai scripted read speech; about 8.38 GB; requires MDC API access after terms acceptance'},
}
human_sources_path = PROJECT_ROOT / 'data' / 'manifests' / 'human_audio_sources.json'
human_sources_path.parent.mkdir(parents=True, exist_ok=True)
human_sources_path.write_text(json.dumps(human_sources, indent=2, ensure_ascii=False))
print('Human audio dataset stage complete.')

[skip] VocalSound: พบไฟล์เสียงแล้ว
[skip] Nonspeech7k train: พบไฟล์เสียงแล้ว
[skip] Nonspeech7k test: พบไฟล์เสียงแล้ว
[needs setup] Common Voice Thai: ยอมรับ terms และสร้าง API key ที่ Mozilla Data Collective แล้วเพิ่ม MDC_API_KEY ใน .env จากนั้นรัน cell นี้ซ้ำ
Human audio dataset stage complete.


## 14. ตรวจรายการเสียงปืนซ้ำ (ไม่ลบไฟล์ต้นฉบับ)
เปรียบเทียบ SHA-256 ข้ามชุดที่ดาวน์โหลด และทำเครื่องหมายคลิป C3GD ที่เป็นเหตุยิงเดียวกันแต่บันทึกคนละไมค์ เพื่อให้ขั้นเตรียมข้อมูลเลือกใช้ชุดไม่ซ้ำได้


In [56]:
from collections import defaultdict

def file_sha256(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''): h.update(chunk)
    return h.hexdigest()

audio_rows = []
for source_name, folder in GUNSHOT_EXTRA_DIRS.items():
    if not folder.exists(): continue
    for path in folder.rglob('*'):
        if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS:
            audio_rows.append({'source':source_name, 'path':str(path.relative_to(PROJECT_ROOT)), 'sha256':file_sha256(path)})
# Exact duplicate audio files across selected new sources.
seen_hash = {}
for row in audio_rows:
    row['duplicate_of'] = seen_hash.get(row['sha256'])
    if row['duplicate_of'] is None: seen_hash[row['sha256']] = row['path']
# C3GD filename: ClassId-EventId-Platform-MicId-FileId-ClipId.wav.
# Same event/clip recorded by different microphones is represented once in the curated manifest.
c3gd_seen = {}
for row in audio_rows:
    path = Path(row['path'])
    if row['source'] == 'c3gd':
        parts = path.stem.split('-')
        if len(parts) >= 6:
            event_key = '-'.join([parts[0], parts[1], parts[2], parts[4], parts[5]])
            row['c3gd_event_key'] = event_key
            row['same_c3gd_event_other_mic'] = c3gd_seen.get(event_key)
            c3gd_seen.setdefault(event_key, row['path'])
        else:
            row['c3gd_event_key'] = ''
            row['same_c3gd_event_other_mic'] = None
    else:
        row['c3gd_event_key'] = ''
        row['same_c3gd_event_other_mic'] = None
    row['unique_candidate'] = row['duplicate_of'] is None and row['same_c3gd_event_other_mic'] is None

dedup_path = PROJECT_ROOT / 'data' / 'manifests' / 'gunshot_extra_dedup_manifest.csv'
dedup_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(audio_rows).to_csv(dedup_path, index=False)
exact_duplicates = sum(r['duplicate_of'] is not None for r in audio_rows)
c3gd_mic_duplicates = sum(r['same_c3gd_event_other_mic'] is not None for r in audio_rows)
print(f'เสียงที่สแกน: {len(audio_rows):,}; SHA ซ้ำ: {exact_duplicates:,}; C3GD เหตุการณ์ซ้ำข้ามไมค์: {c3gd_mic_duplicates:,}')
print('Dedup manifest:', dedup_path.relative_to(PROJECT_ROOT))

เสียงที่สแกน: 15,430; SHA ซ้ำ: 33; C3GD เหตุการณ์ซ้ำข้ามไมค์: 1,276
Dedup manifest: data/manifests/gunshot_extra_dedup_manifest.csv


## 15. สรุปจำนวนและขนาดไฟล์ แล้วบันทึก manifest


In [57]:
from datetime import datetime, timezone
import json
def summarize(folder):
    files = [p for p in folder.rglob('*') if p.is_file() and p.name not in {'.DS_Store', '.gitkeep'}]
    return {'path': str(folder.relative_to(PROJECT_ROOT)), 'file_count': len(files), 'total_bytes': sum(p.stat().st_size for p in files)}
DATASEC_SUMMARY_DIR = RAW_DIR / 'datasec_v2'
if not has_audio(DATASEC_SUMMARY_DIR) and has_audio(RAW_DIR / 'datasec_vehicle_v2'):
    DATASEC_SUMMARY_DIR = RAW_DIR / 'datasec_vehicle_v2'
DATASET_DIRS = {
    'esc50': RAW_DIR / 'esc50', 'footstep_set': RAW_DIR / 'footstep_set',
    'datasec_v2': DATASEC_SUMMARY_DIR, 'rfcx_frugalai': RAW_DIR / 'rfcx_frugalai',
    'fsc22': RAW_DIR / 'fsc22', 'fsd50k_v1': RAW_DIR / 'fsd50k_v1',
    'fire_google_drive': RAW_DIR / 'fire_google_drive', 'zenodo_fire': RAW_DIR / 'zenodo_fire',
    'vehicle_sounds': RAW_DIR / 'vehicle_sounds',
    'c3gd': RAW_DIR / 'c3gd', 'tropical_forest_gunshot': RAW_DIR / 'tropical_forest_gunshot',
    'zenodo_gunshot': RAW_DIR / 'zenodo_gunshot', 'm16_gunshots': RAW_DIR / 'm16_gunshots',
    'vocalsound': RAW_DIR / 'vocalsound', 'nonspeech7k': RAW_DIR / 'nonspeech7k',
    'common_voice_thai_27': RAW_DIR / 'common_voice_thai_27',
}
summary = {name: summarize(path) for name, path in DATASET_DIRS.items()}
manifest = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'datasets': summary}
manifest_path = PROJECT_ROOT / 'data' / 'manifests' / 'download_manifest.json'
manifest_path.parent.mkdir(parents=True, exist_ok=True)
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
for name, info in summary.items(): print(f"{name}: {info['file_count']:,} files, {info['total_bytes'] / (1024**3):.2f} GiB")
print('Manifest:', manifest_path.relative_to(PROJECT_ROOT))


esc50: 2,013 files, 1.42 GiB
footstep_set: 2,129 files, 1.15 GiB
datasec_v2: 5,048 files, 7.01 GiB
rfcx_frugalai: 50,456 files, 14.20 GiB
fsc22: 2,027 files, 1.67 GiB
fsd50k_v1: 0 files, 0.00 GiB
fire_google_drive: 722 files, 0.64 GiB
zenodo_fire: 0 files, 0.00 GiB
vehicle_sounds: 9,448 files, 7.83 GiB
c3gd: 12,138 files, 0.98 GiB
tropical_forest_gunshot: 748 files, 0.04 GiB
zenodo_gunshot: 2,150 files, 1.58 GiB
m16_gunshots: 847 files, 2.11 GiB
vocalsound: 42,030 files, 2.62 GiB
nonspeech7k: 7,017 files, 2.67 GiB
common_voice_thai_27: 0 files, 0.00 GiB
Manifest: data/manifests/download_manifest.json
